# 07 — Attention U-Net 학습 (Phase 6)

**목표**: U-Net 디코더에 SCSE Attention 추가 + LM Heatmap → 최강 조합 검증.

**SCSE Attention** (Spatial-Channel Squeeze & Excitation):
- 공간 차원 + 채널 차원 둘 다에 attention 가중치 학습
- Skip Connection에서 "중요한 영역/채널"만 강조
- 학술 인용: Roy et al., MICCAI 2018

**비교군**:
| 모델 | mIoU |
|------|------|
| Baseline U-Net (final) | 0.6537 |
| + LM Heatmap (best) | 0.6831 |
| ★ + Attention (this run) | ??? |
| ★ + Attention + LM | ??? |

**예상 소요**: 60~80분 (학습 + 캐시 사용)

**⚠️ 환경 주의**:
- mediapipe 설치 시 numpy 충돌 가능 → **새 런타임에서 첫 셀부터** 실행
- 충돌 시 cell 2의 `MODE='attention_only'`로 mediapipe 우회

## 1. 환경 셋업

In [ ]:
!nvidia-smi

In [ ]:
# ★ MODE 설정 — 환경 안정성에 따라 선택 ★
#   'attention_lm'   : Attention + LM Heatmap (최강, mediapipe 필요)
#   'attention_only' : Attention만 (mediapipe 불필요, 안전)
MODE = 'attention_lm'   # 또는 'attention_only'

print(f'학습 모드: {MODE}')

if MODE == 'attention_lm':
    !pip install --quiet segmentation-models-pytorch albumentations datasets pyyaml mediapipe==0.10.21 tqdm
else:
    !pip install --quiet segmentation-models-pytorch albumentations datasets pyyaml tqdm

In [ ]:
import torch, segmentation_models_pytorch as smp
print('torch:', torch.__version__, '  cuda:', torch.cuda.is_available())
print('smp:', smp.__version__)

if MODE == 'attention_lm':
    import mediapipe
    print('mediapipe:', mediapipe.__version__)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

PROJECT_ROOT = '/content/drive/MyDrive/cv-final'
import sys, os
sys.path.insert(0, PROJECT_ROOT)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device)

## 2. Attention U-Net 동작 검증 (sanity check)

In [ ]:
from project.segmentation.unet import build_unet

# Standard vs Attention 파라미터 수 비교
in_ch = 4 if MODE == 'attention_lm' else 3

m_std = build_unet(num_classes=6, in_channels=in_ch, attention_type=None)
m_att = build_unet(num_classes=6, in_channels=in_ch, attention_type='scse')

p_std = sum(p.numel() for p in m_std.parameters()) / 1e6
p_att = sum(p.numel() for p in m_att.parameters()) / 1e6

print(f'Standard U-Net (in={in_ch}): {p_std:.2f}M parameters')
print(f'+ SCSE Attention (in={in_ch}): {p_att:.2f}M parameters  (+{p_att-p_std:.2f}M)')

# Forward 1배치 검증
x = torch.randn(2, in_ch, 256, 256)
y = m_att(x)
print(f'\nForward test: {x.shape} → {y.shape}')
assert y.shape == (2, 6, 256, 256), 'output shape 오류'
print('✅ Attention U-Net 정상 동작')

del m_std, m_att  # 메모리 회수

## 3. 학습 config 작성

In [ ]:
import yaml

CONFIG_PATH = f'{PROJECT_ROOT}/project/segmentation/config.yaml'
with open(CONFIG_PATH, 'r') as f:
    cfg = yaml.safe_load(f)

# Attention U-Net 학습 설정
cfg['dataset']['subset_size'] = None  # 전체 24K
cfg['train']['epochs'] = 20  # Early Stopping이 조기 종료
cfg['output']['checkpoint_dir'] = f'{PROJECT_ROOT}/project/segmentation/checkpoints'
cfg['dataset']['cache_dir'] = '/content/cv-final-cache'
cfg['early_stopping']['enabled'] = True
cfg['early_stopping']['patience'] = 5

# ★ Attention 활성화
cfg['model']['attention_type'] = 'scse'
cfg['model']['use_landmark'] = (MODE == 'attention_lm')

os.makedirs(cfg['output']['checkpoint_dir'], exist_ok=True)

TMP_CONFIG = '/tmp/config_attention.yaml'
with open(TMP_CONFIG, 'w') as f:
    yaml.safe_dump(cfg, f)

print('Attention U-Net config:')
print(f'  attention_type : {cfg["model"]["attention_type"]}')
print(f'  use_landmark   : {cfg["model"]["use_landmark"]}')
print(f'  epochs         : {cfg["train"]["epochs"]}')
print(f'  Early Stopping : patience={cfg["early_stopping"]["patience"]}')

## 4. 학습 실행

**예상 시간**:
- `attention_only`: ~60분
- `attention_lm`: 첫 실행 시 캐시 빌드 (~15분) + 학습 (~70분) = **~85분**
- 캐시 있으면 학습만 ~70분

In [ ]:
from project.segmentation.train import train

result = train(config_path=TMP_CONFIG)
print('\n=== Attention 학습 완료 ===')
print(f"Best mIoU:  {result['best_miou']:.4f} at epoch {result['best_epoch']}")
print(f"Best ckpt: {result['best_ckpt_path']}")
print(f"Model tag: {result['model_tag']}")

## 5. 학습 곡선 + 이전 모델 비교

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

epochs = [h['epoch'] for h in result['history']]
losses = [h['loss'] for h in result['history']]
mious = [h['miou'] for h in result['history']]

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].plot(epochs, losses, marker='o', color='#1E2761')
axes[0].set_title('Attention Training Loss')
axes[0].set_xlabel('epoch'); axes[0].set_ylabel('loss')
axes[0].grid(alpha=0.3)

axes[1].plot(epochs, mious, marker='o', color='#F96167', label=f'Attention ({MODE})')
axes[1].axhline(y=0.6537, color='gray', linestyle='--', label='Baseline final (0.6537)')
axes[1].axhline(y=0.6831, color='blue', linestyle='--', label='LM-guided (0.6831)')
axes[1].set_title('Validation mIoU')
axes[1].set_xlabel('epoch'); axes[1].set_ylabel('mIoU')
axes[1].grid(alpha=0.3); axes[1].legend()
plt.tight_layout()

os.makedirs(f'{PROJECT_ROOT}/data/outputs', exist_ok=True)
plt.savefig(f'{PROJECT_ROOT}/data/outputs/attention_training_curve.png', dpi=150)
plt.show()

## 6. 예측 시각화 — Attention 효과 확인

In [ ]:
from project.segmentation import CelebAMaskHQDataset, get_val_transform

use_lm = (MODE == 'attention_lm')

model = build_unet(
    num_classes=6, in_channels=(4 if use_lm else 3),
    encoder_weights=None, attention_type='scse',
)
model.load_state_dict(torch.load(result['best_ckpt_path'], map_location=device))
model.to(device).eval()

ds_viz = CelebAMaskHQDataset(
    split='val',
    transform=get_val_transform(size=256, with_heatmap=use_lm),
    subset_size=5, cache_dir='/content/cv-final-cache',
    use_landmark=use_lm,
)

fig, axes = plt.subplots(5, 3, figsize=(12, 18))
for i in range(5):
    img, gt_mask = ds_viz[i]
    rgb = img[:3].permute(1, 2, 0).numpy()
    rgb = rgb * np.array([0.229, 0.224, 0.225]) + np.array([0.485, 0.456, 0.406])
    rgb = rgb.clip(0, 1)
    with torch.no_grad():
        pred = model(img.unsqueeze(0).to(device)).argmax(dim=1).squeeze(0).cpu().numpy()
    axes[i, 0].imshow(rgb); axes[i, 0].set_title('Input'); axes[i, 0].axis('off')
    axes[i, 1].imshow(gt_mask.numpy(), cmap='tab10', vmin=0, vmax=5)
    axes[i, 1].set_title('Ground Truth'); axes[i, 1].axis('off')
    axes[i, 2].imshow(pred, cmap='tab10', vmin=0, vmax=5)
    axes[i, 2].set_title(f'Attention U-Net ★ ({MODE})'); axes[i, 2].axis('off')

plt.tight_layout()
plt.savefig(f'{PROJECT_ROOT}/data/outputs/attention_predictions.png', dpi=150)
plt.show()

## 7. 최종 비교표 (Baseline / LM / Attention)

In [ ]:
BASELINE_FINAL = 0.6537   # Phase 1 (final, overfit)
BASELINE_BEST_KNOWN = 0.680  # Phase 1 training graph 정점
LM_BEST = 0.6831         # Phase 3
ATTENTION_BEST = result['best_miou']

print('═' * 76)
print('   강화 단계별 누적 효과 (Val mIoU)')
print('═' * 76)
print(f'  Baseline U-Net (20 epoch, overfit final) :  {BASELINE_FINAL:.4f}')
print(f'  Baseline U-Net (best at epoch ~7)        :  ~{BASELINE_BEST_KNOWN:.4f}')
print(f'  + LM Heatmap (Early Stop, Phase 3)       :  {LM_BEST:.4f}   (Δ {(LM_BEST-BASELINE_FINAL)*100:+.2f}%p)')
print(f'  ★ + Attention ({MODE})                   :  {ATTENTION_BEST:.4f}   (Δ {(ATTENTION_BEST-BASELINE_FINAL)*100:+.2f}%p)')
print('═' * 76)

att_gain_vs_lm = (ATTENTION_BEST - LM_BEST) * 100
total_gain = (ATTENTION_BEST - BASELINE_FINAL) * 100

print(f'\nAttention 추가 효과 (vs LM):  {att_gain_vs_lm:+.2f}%p')
print(f'Pipeline 통합 효과 (vs Baseline final):  {total_gain:+.2f}%p')

if att_gain_vs_lm >= 1.0:
    print('\n  ✅ Attention 효과 큼 — 발표 자료 확정')
elif att_gain_vs_lm >= 0.3:
    print('\n  ⚠️ Attention 효과 보통 — 추가 TTA 적용 검토')
else:
    print('\n  🟡 Attention 효과 미미 — Streamlit 통합으로 전환')

## 8. 완료 체크리스트 (Phase 6)

- [ ] MODE 선택 (`attention_lm` 또는 `attention_only`)
- [ ] Attention U-Net 파라미터 수 증가 확인 (~22M → ~23M)
- [ ] 1 batch forward sanity check 성공
- [ ] Early Stopping으로 학습 완료
- [ ] Best checkpoint 저장 (`unet_attention*_best.pt`)
- [ ] 비교표 출력 (Baseline / LM / Attention)
- [ ] 예측 시각화 5장 저장

**다음 단계 (Phase 7 — 선택)**:
- Attention + TTA 측정 → 최종 최강 조합
- 또는 발표 자료 확정 + Streamlit 진입